# Predictive Analytics Using Historical DataProject #3 — Thiranex Skill Development

## Step 1: Imports

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltfrom sklearn.linear_model import LinearRegressionfrom sklearn.metrics import mean_squared_error, mean_absolute_error, r2_scoreprint("All imports successful!")

## Step 2: Generate/load the dataset

In [ ]:
np.random.seed(42)dates = pd.date_range(start='2023-01-01', periods=48, freq='MS')trend = np.linspace(1000, 2200, len(dates))seasonality = 300 * np.sin(2 * np.pi * dates.month / 12)noise = np.random.normal(0, 80, len(dates))sales = np.round(trend + seasonality + noise, 2)df = pd.DataFrame({'Date': dates, 'Sales': sales})df.loc[5, 'Sales'] = np.nandf.loc[20, 'Sales'] = np.nandf = pd.concat([df, df.iloc[[10]]], ignore_index=True)df.to_csv('../data/raw/sales_data.csv', index=False)print(df.head(10))

## Step 3: Clean and preprocess

In [ ]:
df = pd.read_csv('../data/raw/sales_data.csv', parse_dates=['Date'])df = df.drop_duplicates()df = df.sort_values('Date').reset_index(drop=True)df['Sales'] = df['Sales'].interpolate(method='linear')df.set_index('Date', inplace=True)df.to_csv('../data/processed/sales_data_cleaned.csv')print(df.head(10))

## Step 4: Exploratory data analysis (EDA)

In [ ]:
plt.figure(figsize=(12, 6))plt.plot(df.index, df['Sales'], marker='o', linewidth=2, color='#2563eb')plt.title('Monthly Sales Trend')plt.xlabel('Date')plt.ylabel('Sales')plt.grid(True, alpha=0.3)plt.tight_layout()plt.savefig('../outputs/figures/sales_trend.png')plt.show()df['Month'] = df.index.monthprint(df.groupby('Month')['Sales'].mean())

## Step 5: Feature engineering

In [ ]:
df = df.reset_index()df['Sales_Lag1'] = df['Sales'].shift(1)df['Sales_Lag2'] = df['Sales'].shift(2)df['Rolling_Mean_3'] = df['Sales'].shift(1).rolling(window=3).mean()  # shifted to avoid leakagedf['Month'] = df['Date'].dt.monthdf['Year'] = df['Date'].dt.yeardf['Time_Index'] = np.arange(len(df))df_model = df.dropna().reset_index(drop=True)print(df_model.head(10))print("\nShape after feature engineering:", df_model.shape)

## Step 6: Train the model

In [ ]:
features = ['Time_Index', 'Month', 'Sales_Lag1', 'Sales_Lag2', 'Rolling_Mean_3']X = df_model[features]y = df_model['Sales']split_index = int(len(df_model) * 0.8)X_train, X_test = X[:split_index], X[split_index:]y_train, y_test = y[:split_index], y[split_index:]model = LinearRegression()model.fit(X_train, y_train)y_pred = model.predict(X_test)comparison = pd.DataFrame({'Actual': y_test.values, 'Predicted': y_pred.round(2)})print(comparison)

## Step 7: Evaluate accuracy

In [ ]:
mae = mean_absolute_error(y_test, y_pred)rmse = np.sqrt(mean_squared_error(y_test, y_pred))r2 = r2_score(y_test, y_pred)print(f"MAE:  {mae:.2f}")print(f"RMSE: {rmse:.2f}")print(f"R2:   {r2:.4f}")avg_sales = y_test.mean()print(f"Error as % of average sales: {(mae/avg_sales)*100:.2f}%")

## Step 8: Visualize predictions and forecast

In [ ]:
plt.figure(figsize=(12, 6))test_dates = df_model['Date'].iloc[split_index:]plt.plot(test_dates, y_test.values, marker='o', label='Actual', color='#2563eb', linewidth=2)plt.plot(test_dates, y_pred, marker='s', label='Predicted', color='#dc2626', linewidth=2, linestyle='--')plt.title('Actual vs Predicted Sales (Test Set)')plt.xlabel('Date'); plt.ylabel('Sales'); plt.legend(); plt.grid(True, alpha=0.3)plt.tight_layout()plt.savefig('../outputs/figures/actual_vs_predicted.png')plt.show()last_sales = list(df_model['Sales'].iloc[-3:])future_dates = pd.date_range(start=df_model['Date'].max() + pd.DateOffset(months=1), periods=6, freq='MS')future_df = pd.DataFrame({'Date': future_dates})future_df['Month'] = future_df['Date'].dt.monthfuture_df['Time_Index'] = np.arange(len(df_model), len(df_model) + 6)future_preds = []for i in range(6):    lag1, lag2 = last_sales[-1], last_sales[-2]    roll3 = np.mean(last_sales[-3:])    Xf = pd.DataFrame([[future_df['Time_Index'].iloc[i], future_df['Month'].iloc[i], lag1, lag2, roll3]], columns=features)    pred = model.predict(Xf)[0]    future_preds.append(pred)    last_sales.append(pred)future_df['Forecast'] = future_predsprint(future_df[['Date', 'Forecast']])plt.figure(figsize=(12, 6))plt.plot(df_model['Date'], df_model['Sales'], label='Historical', color='#2563eb', linewidth=2)plt.plot(future_df['Date'], future_df['Forecast'], label='Forecast', color='#16a34a', linewidth=2, linestyle='--', marker='o')plt.axvline(x=df_model['Date'].max(), color='gray', linestyle=':', alpha=0.5)plt.title('Sales Forecast — Next 6 Months')plt.xlabel('Date'); plt.ylabel('Sales'); plt.legend(); plt.grid(True, alpha=0.3)plt.tight_layout()plt.savefig('../outputs/figures/forecast.png')plt.show()

## Step 9: Generate final report

In [ ]:
report = f"""# Predictive Analytics Using Historical Data — Project Report## 1. ObjectiveBuild a predictive model to forecast future sales trends using historical data,applying regression-based time-series forecasting techniques.## 2. Dataset- 48 months of historical sales data (Jan 2023 – Dec 2026)- Columns: Date, Sales- Contained 2 missing values and 1 duplicate row (realistic messiness)## 3. Data Cleaning- Removed duplicate rows- Sorted chronologically- Filled missing values using linear interpolation- Set Date as the time-series index## 4. Exploratory Data Analysis- Identified an overall upward trend in sales over the 4-year period- Detected seasonal fluctuations across months (sine-wave pattern)## 5. Feature Engineering- Sales_Lag1, Sales_Lag2: previous 1-2 months' sales- Rolling_Mean_3: 3-month moving average (using only past data, no leakage)- Time_Index: sequential counter to capture overall trend- Month: captures seasonality## 6. Model- Algorithm: Linear Regression (scikit-learn)- Train/test split: 80/20, chronological## 7. Evaluation Metrics- MAE: {mae:.2f}- RMSE: {rmse:.2f}- R2 Score: {r2:.4f}## 8. ConclusionThe model explains approximately {r2*100:.0f}% of the variance in monthly sales,with an average prediction error of about {(mae/avg_sales)*100:.1f}%."""with open('../reports/final_report.md', 'w') as f:    f.write(report)print(report)